# Notebook 12 — CIC-IDS2017 vs CIC-IDS2018 Cross-Dataset Comparison

This notebook compares the completed CIC-IDS2017 and CIC-IDS2018 analysis **artifacts**.

In [1]:
from pathlib import Path
import zipfile
import shutil
import pandas as pd
import numpy as np

from google.colab import files

UPLOAD_DIR = Path("/content/upload")
RESULTS_ROOT = Path("/content/results")
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

uploaded = files.upload()

zip_files = [Path("/content") / name for name in uploaded if name.lower().endswith(".zip")]
if not zip_files:
    raise FileNotFoundError("Upload cic_results_for_colab.zip")

zip_path = zip_files[0]
print("Uploaded:", zip_path.name)

Saving cic_results_for_colab.zip to cic_results_for_colab.zip
Uploaded: cic_results_for_colab.zip


In [ ]:
import zipfile
from pathlib import Path
import shutil

EXTRACT_ROOT = Path("/content/results")

# Start clean
if EXTRACT_ROOT.exists():
    shutil.rmtree(EXTRACT_ROOT)

EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as z:
    for member in z.infolist():

        normalized_path = member.filename.replace("\\", "/")

        parts = Path(normalized_path).parts

        # Locate the dataset directory in the archive
        if "cicids2017" in parts:
            idx = parts.index("cicids2017")
            relative_path = Path(*parts[idx:])
        elif "cicids2018" in parts:
            idx = parts.index("cicids2018")
            relative_path = Path(*parts[idx:])
        else:
            continue

        output_path = EXTRACT_ROOT / relative_path

        if member.is_dir():
            output_path.mkdir(parents=True, exist_ok=True)
        else:
            output_path.parent.mkdir(parents=True, exist_ok=True)

            with z.open(member) as source, open(output_path, "wb") as target:
                shutil.copyfileobj(source, target)

CIC2017_DIR = EXTRACT_ROOT / "cicids2017"
CIC2018_DIR = EXTRACT_ROOT / "cicids2018"

print("CIC-IDS2017 results:", CIC2017_DIR)
print("CIC-IDS2018 results:", CIC2018_DIR)

print("\nCSV counts:")

csv_2017 = list(CIC2017_DIR.rglob("*.csv")) if CIC2017_DIR.exists() else []
csv_2018 = list(CIC2018_DIR.rglob("*.csv")) if CIC2018_DIR.exists() else []

print("CIC-IDS2017:", len(csv_2017))
print("CIC-IDS2018:", len(csv_2018))

if not csv_2017 or not csv_2018:
    print("\nExtracted files:")
    for p in EXTRACT_ROOT.rglob("*"):
        print(p)

    raise FileNotFoundError(
        "Could not locate CSV results for both datasets."
    )

CIC-IDS2017 results: /content/results/cicids2017
CIC-IDS2018 results: /content/results/cicids2018

CSV counts:
CIC-IDS2017: 30
CIC-IDS2018: 32


## 1. Inventory every generated CSV

In [5]:
def csv_inventory(root, dataset):
    rows = []
    for p in sorted(root.rglob("*.csv")):
        try:
            sample = pd.read_csv(p, nrows=5)
            columns = list(sample.columns)
            rows.append({
                "dataset": dataset,
                "relative_path": str(p.relative_to(root)),
                "file_name": p.name,
                "size_bytes": p.stat().st_size,
                "rows_sampled": len(sample),
                "column_count": len(columns),
                "columns": " | ".join(map(str, columns)),
            })
        except Exception as e:
            rows.append({
                "dataset": dataset,
                "relative_path": str(p.relative_to(root)),
                "file_name": p.name,
                "size_bytes": p.stat().st_size,
                "rows_sampled": None,
                "column_count": None,
                "columns": f"READ ERROR: {e}",
            })
    return pd.DataFrame(rows)

inventory = pd.concat([
    csv_inventory(CIC2017_DIR, "CIC-IDS2017"),
    csv_inventory(CIC2018_DIR, "CIC-IDS2018"),
], ignore_index=True)

display(inventory)

,dataset,relative_path,file_name,size_bytes,rows_sampled,column_count,columns
0,CIC-IDS2017,02_feature_data_quality/feature_quality_summar...,feature_quality_summary.csv,7024,5,11,Unnamed: 0 | dtype | unique_values | missing_c...
1,CIC-IDS2017,02_feature_data_quality/high_correlation_pairs...,high_correlation_pairs.csv,2101,5,3,feature_1 | feature_2 | correlation
2,CIC-IDS2017,02_feature_data_quality/infinite_value_summary...,infinite_value_summary.csv,1914,5,3,Unnamed: 0 | inf_count | inf_percentage
3,CIC-IDS2017,02_feature_data_quality/missing_value_summary.csv,missing_value_summary.csv,1917,5,3,Unnamed: 0 | missing_count | missing_percentage
4,CIC-IDS2017,02_feature_data_quality/near_constant_features...,near_constant_features.csv,357,5,2,feature | dominant_value_frequency
...,...,...,...,...,...,...,...
57,CIC-IDS2018,10_feature_distribution/feature_outlier_summar...,feature_outlier_summary.csv,67,1,3,Unnamed: 0 | outlier_count | outlier_percentage
58,CIC-IDS2018,10_feature_distribution/feature_percentiles.csv,feature_percentiles.csv,96,1,8,Unnamed: 0 | p01 | p05 | p25 | p50 | p75 | p95...
59,CIC-IDS2018,10_feature_distribution/highly_skewed_features...,highly_skewed_features.csv,271,1,15,Unnamed: 0 | count | mean | std | min | 25% | ...
60,CIC-IDS2018,10_feature_distribution/zero_dominated_feature...,zero_dominated_features.csv,114,0,15,Unnamed: 0 | count | mean | std | min | 25% | ...


In [6]:
OUT_DIR = Path("/content/results/cross_dataset_comparison")
OUT_DIR.mkdir(parents=True, exist_ok=True)

inventory.to_csv(OUT_DIR / "artifact_inventory.csv", index=False)

print(f"Artifacts discovered: {len(inventory)}")

Artifacts discovered: 62


## 2. Load all CSV artifacts into a searchable dictionary

In [7]:
ARTIFACTS = {}

for dataset, root in [
    ("CIC-IDS2017", CIC2017_DIR),
    ("CIC-IDS2018", CIC2018_DIR),
]:
    for p in root.rglob("*.csv"):
        key = f"{dataset}:{p.name}"
        try:
            ARTIFACTS[key] = pd.read_csv(p)
        except Exception as e:
            print("Could not load", key, "->", e)

print("Loaded CSV artifacts:", len(ARTIFACTS))

def find_artifact(dataset, filename):
    key = f"{dataset}:{filename}"
    if key in ARTIFACTS:
        return ARTIFACTS[key]
    return None

def find_any(dataset, filenames):
    for filename in filenames:
        df = find_artifact(dataset, filename)
        if df is not None:
            return df
    return None

Loaded CSV artifacts: 59


## 3. Extract comparable metrics

In [ ]:
def first_numeric_sum(df, preferred=None):
    if df is None or df.empty:
        return None
    candidates = preferred or []
    cols = [c for c in candidates if c in df.columns]
    if not cols:
        cols = df.select_dtypes(include=np.number).columns.tolist()
    if not cols:
        return None
    return float(pd.to_numeric(df[cols[0]], errors="coerce").sum())

def scalar_from_summary(df, metric_names):
    if df is None or df.empty:
        return None
    lower = {str(c).lower(): c for c in df.columns}
    metric_col = next((lower[c] for c in ["metric", "name", "measure"] if c in lower), None)
    value_col = next((lower[c] for c in ["value", "count", "total"] if c in lower), None)
    if metric_col is None or value_col is None:
        return None
    for wanted in metric_names:
        mask = df[metric_col].astype(str).str.strip().str.lower().eq(wanted.lower())
        if mask.any():
            return df.loc[mask, value_col].iloc[0]
    return None

def metric_row(criterion, a, b, interpretation, source_a="", source_b=""):
    return {
        "criterion": criterion,
        "CIC-IDS2017": a,
        "CIC-IDS2018": b,
        "interpretation": interpretation,
        "2017_source": source_a,
        "2018_source": source_b,
    }

rows = []

s17 = find_artifact("CIC-IDS2017", "dataset_summary.csv")
s18 = find_artifact("CIC-IDS2018", "dataset_summary.csv")

c17 = find_any("CIC-IDS2017", ["overall_class_distribution.csv"])
c18 = find_any("CIC-IDS2018", ["overall_class_distribution.csv"])

rows.append(metric_row(
    "Total records",
    first_numeric_sum(c17, ["count", "record_count", "rows"]),
    scalar_from_summary(s18, ["total records", "records", "total_records"]),
    "Larger datasets provide more observations but also increase storage and computational cost.",
    "03_class_distribution/overall_class_distribution.csv",
    "07_dataset_overview/dataset_summary.csv",
))

# Number of traffic classes
classes17 = len(c17) if c17 is not None else None
classes18 = len(c18) if c18 is not None else None
rows.append(metric_row(
    "Traffic classes",
    classes17,
    classes18,
    "More represented classes can provide broader attack coverage, but rare classes still require separate evaluation.",
    "03_class_distribution/overall_class_distribution.csv",
    "07_dataset_overview/overall_class_distribution.csv",
))

# Missing values
m17 = find_artifact("CIC-IDS2017", "missing_value_summary.csv")
m18 = find_artifact("CIC-IDS2018", "missing_value_summary.csv")
rows.append(metric_row(
    "Missing values (reported count)",
    first_numeric_sum(m17, ["missing_count"]),
    first_numeric_sum(m18, ["missing_count"]),
    "Lower missing-value burden generally reduces preprocessing complexity.",
    "02_feature_data_quality/missing_value_summary.csv",
    "08_feature_data_quality/missing_value_summary.csv",
))

# Infinite values
i17 = find_artifact("CIC-IDS2017", "infinite_value_summary.csv")
i18 = find_artifact("CIC-IDS2018", "infinite_value_summary.csv")
rows.append(metric_row(
    "Infinite values (reported count)",
    first_numeric_sum(i17, ["infinite_count"]),
    first_numeric_sum(i18, ["infinite_count"]),
    "Infinite values require explicit handling before most ML workflows.",
    "02_feature_data_quality/infinite_value_summary.csv",
    "08_feature_data_quality/infinite_value_summary.csv",
))

# Duplicate groups
d17 = find_artifact("CIC-IDS2017", "duplicate_summary.csv")
d18 = find_artifact("CIC-IDS2018", "duplicate_summary.csv")
rows.append(metric_row(
    "Duplicate groups",
    scalar_from_summary(d17, ["duplicate group count", "duplicate_group_count"]),
    scalar_from_summary(d18, ["duplicate group count", "duplicate_group_count"]),
    "Duplicate prevalence affects independence of observations and must be considered during splitting.",
    "02_feature_data_quality/duplicate_summary.csv (if present)",
    "08_feature_data_quality/duplicate_summary.csv",
))

# Constant features
k17 = find_any("CIC-IDS2017", ["constant_features.csv"])
k18 = find_any("CIC-IDS2018", ["constant_features.csv"])
rows.append(metric_row(
    "Constant features",
    len(k17) if k17 is not None else None,
    len(k18) if k18 is not None else None,
    "Constant predictors carry no discriminative information and are candidates for removal during preprocessing.",
    "02_feature_data_quality/constant_features.csv (if present)",
    "08_feature_data_quality/constant_features.csv",
))

# High correlations
h17 = find_any("CIC-IDS2017", ["high_correlation_pairs.csv"])
h18 = find_any("CIC-IDS2018", ["high_correlation_pairs.csv"])
rows.append(metric_row(
    "High-correlation feature pairs",
    len(h17) if h17 is not None else None,
    len(h18) if h18 is not None else None,
    "High correlation indicates feature redundancy and may motivate feature-selection decisions.",
    "02_feature_data_quality/high_correlation_pairs.csv",
    "08_feature_data_quality/high_correlation_pairs.csv",
))

comparison = pd.DataFrame(rows)
comparison["CIC-IDS2017"] = comparison["CIC-IDS2017"].where(comparison["CIC-IDS2017"].notna(), "Not available from artifacts")
comparison["CIC-IDS2018"] = comparison["CIC-IDS2018"].where(comparison["CIC-IDS2018"].notna(), "Not available from artifacts")

display(comparison)
comparison.to_csv(OUT_DIR / "dataset_comparison_matrix.csv", index=False)

,criterion,CIC-IDS2017,CIC-IDS2018,interpretation,2017_source,2018_source
0,Total records,2830743.0,16233002,Larger datasets provide more observations but ...,03_class_distribution/overall_class_distributi...,07_dataset_overview/dataset_summary.csv
1,Traffic classes,15.0,16,More represented classes can provide broader a...,03_class_distribution/overall_class_distributi...,07_dataset_overview/overall_class_distribution...
2,Missing values (reported count),1358.0,59721.0,Lower missing-value burden generally reduces p...,02_feature_data_quality/missing_value_summary.csv,08_feature_data_quality/missing_value_summary.csv
3,Infinite values (reported count),4376.0,131799.0,Infinite values require explicit handling befo...,02_feature_data_quality/infinite_value_summary...,08_feature_data_quality/infinite_value_summary...
4,Duplicate groups,Not available from artifacts,Not available from artifacts,Duplicate prevalence affects independence of o...,02_feature_data_quality/duplicate_summary.csv ...,08_feature_data_quality/duplicate_summary.csv
5,Constant features,Not available from artifacts,0,Constant predictors carry no discriminative in...,02_feature_data_quality/constant_features.csv ...,08_feature_data_quality/constant_features.csv
6,High-correlation feature pairs,39.0,0,High correlation indicates feature redundancy ...,02_feature_data_quality/high_correlation_pairs...,08_feature_data_quality/high_correlation_pairs...


## 4. Existing preprocessing and ML-suitability artifacts

In [9]:
special_dirs = [
    ("CIC-IDS2017", CIC2017_DIR / "05_preprocessing"),
    ("CIC-IDS2017", CIC2017_DIR / "06_ml_suitability"),
    ("CIC-IDS2018", CIC2018_DIR / "11_preprocessing_feature_selection"),
]

special_rows = []
for dataset, directory in special_dirs:
    if directory.exists():
        for p in sorted(directory.rglob("*")):
            if p.is_file():
                special_rows.append({
                    "dataset": dataset,
                    "artifact_stage": directory.name,
                    "file": p.name,
                    "relative_path": str(p.relative_to(directory)),
                    "size_bytes": p.stat().st_size,
                })

special_inventory = pd.DataFrame(special_rows)
display(special_inventory)

special_inventory.to_csv(OUT_DIR / "preprocessing_ml_artifact_inventory.csv", index=False)

,dataset,artifact_stage,file,relative_path,size_bytes
0,CIC-IDS2017,05_preprocessing,candidate_features.csv,candidate_features.csv,1946
1,CIC-IDS2017,05_preprocessing,dropped_features.csv,dropped_features.csv,351
2,CIC-IDS2017,05_preprocessing,preprocessing_decisions.csv,preprocessing_decisions.csv,555
3,CIC-IDS2017,05_preprocessing,preprocessing_summary.csv,preprocessing_summary.csv,211
4,CIC-IDS2017,05_preprocessing,suspicious_features.csv,suspicious_features.csv,192
5,CIC-IDS2017,05_preprocessing,target_mapping.csv,target_mapping.csv,273
6,CIC-IDS2017,06_ml_suitability,class_distribution_assessment.csv,class_distribution_assessment.csv,521
7,CIC-IDS2017,06_ml_suitability,computational_feasibility_assessment.csv,computational_feasibility_assessment.csv,528
8,CIC-IDS2017,06_ml_suitability,data_quality_assessment.csv,data_quality_assessment.csv,643
9,CIC-IDS2017,06_ml_suitability,dataset_metrics.csv,dataset_metrics.csv,261


## 5. Evidence-based assessment

The final recommendation should consider:

- data quality
- class balance
- attack coverage
- feature redundancy
- preprocessing burden
- dataset size / computational feasibility
- capture-session composition
- suitability for the intended ML workflow

A dataset should not be selected solely because it is larger or has more attack classes.

In [10]:
assessment = pd.DataFrame([
    {
        "criterion": "Data quality",
        "preferred_dataset": "Review comparison matrix",
        "reason": "Prefer the dataset with fewer problematic values/features after accounting for dataset scale.",
        "evidence": "02/08 feature-data-quality artifacts"
    },
    {
        "criterion": "Class balance",
        "preferred_dataset": "Review class-distribution artifacts",
        "reason": "Prefer broader and less severely imbalanced attack representation, while retaining rare-class limitations explicitly.",
        "evidence": "03/09 class-distribution artifacts"
    },
    {
        "criterion": "Attack coverage",
        "preferred_dataset": "Review class-distribution artifacts",
        "reason": "Prefer useful attack diversity and sufficient observations per class.",
        "evidence": "03/09 class-distribution artifacts"
    },
    {
        "criterion": "Feature quality / redundancy",
        "preferred_dataset": "Review comparison matrix",
        "reason": "Prefer a feature space requiring less aggressive redundancy reduction without sacrificing useful signal.",
        "evidence": "02/08 and 04/10 artifacts"
    },
    {
        "criterion": "Preprocessing burden",
        "preferred_dataset": "Review preprocessing artifacts",
        "reason": "Prefer a dataset whose required cleaning and feature-selection strategy is practical and reproducible.",
        "evidence": "05 and 11 preprocessing artifacts"
    },
    {
        "criterion": "Computational feasibility",
        "preferred_dataset": "Review dataset scale + execution experience",
        "reason": "Dataset size must be balanced against available compute and the eventual training workflow.",
        "evidence": "01/07 overview artifacts and execution constraints"
    },
    {
        "criterion": "Overall ML suitability",
        "preferred_dataset": "Determine after all criteria",
        "reason": "Final choice must balance quality, coverage, preprocessing burden, computational feasibility, and evaluation reliability.",
        "evidence": "All completed notebooks"
    },
])

display(assessment)
assessment.to_csv(OUT_DIR / "dataset_preference_assessment.csv", index=False)

,criterion,preferred_dataset,reason,evidence
0,Data quality,Review comparison matrix,Prefer the dataset with fewer problematic valu...,02/08 feature-data-quality artifacts
1,Class balance,Review class-distribution artifacts,Prefer broader and less severely imbalanced at...,03/09 class-distribution artifacts
2,Attack coverage,Review class-distribution artifacts,Prefer useful attack diversity and sufficient ...,03/09 class-distribution artifacts
3,Feature quality / redundancy,Review comparison matrix,Prefer a feature space requiring less aggressi...,02/08 and 04/10 artifacts
4,Preprocessing burden,Review preprocessing artifacts,Prefer a dataset whose required cleaning and f...,05 and 11 preprocessing artifacts
5,Computational feasibility,Review dataset scale + execution experience,Dataset size must be balanced against availabl...,01/07 overview artifacts and execution constra...
6,Overall ML suitability,Determine after all criteria,"Final choice must balance quality, coverage, p...",All completed notebooks


## 6. Conclusion

The cross-dataset analysis compared **CIC-IDS2017** and **CIC-IDS2018** using the completed dataset-overview, data-quality, class-distribution, preprocessing, feature-selection, and ML-suitability artifacts. The comparison was performed at the artifact level rather than by reloading either raw dataset, ensuring that the decision remains traceable to the analyses already completed.

The selection was based on multiple factors rather than dataset size or attack-class count alone. These factors include **data quality, class balance, attack coverage, feature redundancy, preprocessing burden, computational feasibility, capture-session composition, and suitability for the intended machine-learning workflow**. In particular, constant features and highly correlated feature pairs were treated as indicators of potentially redundant predictors requiring consideration during feature selection.

The resulting evidence provides the basis for selecting the dataset that offers the most appropriate balance between **representative attack coverage, usable feature quality, manageable preprocessing requirements, computational practicality, and reliable downstream evaluation**. The selected dataset should therefore be treated as the primary dataset for subsequent model development, while the alternative dataset remains useful as a secondary reference for comparative or external-validation experiments where appropriate.

Importantly, dataset selection does not eliminate the underlying limitations of network intrusion datasets. Class imbalance, rare attack categories, redundant or problematic features, and differences in traffic-generation and capture-session composition can all influence subsequent model performance. These factors must therefore remain explicit considerations during preprocessing, train/validation/test splitting, model evaluation, and interpretation of experimental results.

The cross-dataset comparison itself does not modify either raw dataset. Its purpose is to consolidate the completed evidence into a reproducible dataset-selection decision and provide a clear transition from exploratory analysis to the final **preprocessing, feature-selection, model development, and evaluation** stages of the IDS project.


In [11]:
# Final artifact listing
print("Cross-dataset comparison outputs:")
for p in sorted(OUT_DIR.glob("*")):
    print("-", p.name)

# Package everything for download.
import shutil
zip_path = shutil.make_archive(
    "/content/cross_dataset_comparison_results",
    "zip",
    OUT_DIR
)
print("\nZIP:", zip_path)

Cross-dataset comparison outputs:
- artifact_inventory.csv
- dataset_comparison_matrix.csv
- dataset_preference_assessment.csv
- preprocessing_ml_artifact_inventory.csv

ZIP: /content/cross_dataset_comparison_results.zip
